# TEED-Curves: Colab training

Clone the repository, prepare BIPED/BSDS500/CurveML, validate every path and stage, then run the complete three-stage curriculum.


In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO = Path("/content/Lines-curves")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/Apache0ne/Lines-curves.git", str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "compileall", "-q", "."], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
subprocess.run([sys.executable, "scripts/smoke_test.py"], check=True)


## Optional Google Drive output

Keep this enabled to preserve checkpoints if the Colab runtime disconnects.


In [ ]:
USE_GOOGLE_DRIVE = True
CONFIG = Path("configs/colab.yaml")

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    import yaml

    drive.mount("/content/drive")
    config = yaml.safe_load(CONFIG.read_text(encoding="utf-8"))
    config["common"]["output_root"] = "/content/drive/MyDrive/Lines-curves-outputs"
    CONFIG = Path("configs/colab_drive.yaml")
    CONFIG.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")

print(f"CONFIG={CONFIG.resolve()}")


## Download and normalize datasets

`WITH_CURVEML=True` clones the CurveML repository. Set it to `False` to use the built-in exact procedural generator plus natural BIPED/BSDS500 records.


In [ ]:
WITH_CURVEML = True
setup = [sys.executable, "scripts/colab_setup.py"]
if WITH_CURVEML:
    setup.append("--with-curveml")
subprocess.run(setup, check=True)


## Mandatory preflight

This validates the edited config, data counts, checkpoint compatibility, model size, and a finite forward/backward sample for all three stages.


In [ ]:
preflight_report = Path("outputs/preflight.json")
subprocess.run([
    sys.executable, "scripts/preflight.py",
    "--config", str(CONFIG),
    "--download-teed",
    "--report", str(preflight_report),
], check=True)
print(preflight_report.read_text(encoding="utf-8"))


## Train Stage 1 → Stage 2 → Stage 3


In [ ]:
subprocess.run([sys.executable, "train_all.py", "--config", str(CONFIG), "--auto-resume"], check=True)


## Inspect final files and checksums


In [ ]:
import hashlib
import yaml

config = yaml.safe_load(CONFIG.read_text(encoding="utf-8"))
output_root = Path(config["common"]["output_root"])
final_dir = output_root / "stage3"
for path in sorted(final_dir.glob("*")):
    if path.is_file():
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        print(f"{path.name:28s} {path.stat().st_size:12d} bytes  sha256={digest}")


## Resume example

Use the stage number and that stage's `last.pt`. The trainer restores optimizer, scheduler, AMP scaler, and recorded RNG state.


In [ ]:
# Example only; uncomment after an interrupted Stage 1 run.
# subprocess.run([
#     sys.executable, "train.py",
#     "--stage", "1",
#     "--config", str(CONFIG),
#     "--resume", str(output_root / "stage1" / "last.pt"),
# ], check=True)


## ONNX export


In [ ]:
subprocess.run([
    sys.executable, "export.py",
    "--checkpoint", str(final_dir / "best.safetensors"),
    "--output", str(output_root / "teed_curves.onnx"),
], check=True)
